<a href="https://colab.research.google.com/github/shristeca/SETU---Data-Handling-and-Infrastructure-/blob/main/Project_AI_Infra_Object_storage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Project AI Infra - Object storage

## Architecture overview
Below is a flow of the process planned on following for this project.
```
Kaggle CSVs
      ↓
GCS Raw Layer
      ↓
BigQuery Raw Tables
      ↓
SQL Joins & Transformations
      ↓
BigQuery Processed Table
      ↓
Export to GCS Processed Layer (Parquet)
      ↓
Train/Dev/Test Split
      ↓
ML Model
```

### Install all necessary Libraries

In [ ]:
# Install required libraries
!pip install --upgrade google-cloud-storage google-cloud-bigquery pandas scikit-learn requests gcsfs sqlalchemy psycopg2-binary

### Creating a GCS bucket

In [ ]:
import os
import requests
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from google.cloud import storage
from google.cloud import bigquery
from sqlalchemy import create_engine

# --- GCP Configuration ---
PROJECT_ID = "ai-infra-credit-risk"
BUCKET_NAME = "credit-risk-raw-data"
LOCATION = "EU"


# --- Authentication options ---
# Option A: Google Colab / Vertex AI:
from google.colab import auth
auth.authenticate_user()

# Option B: local development with Application Default Credentials:
# Run in a terminal before starting the notebook:
#   gcloud auth application-default login

# Option C: local service-account key (avoid committing this path/key):
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/path/to/service-account-key.json"

print(f"Target project: {PROJECT_ID}")
print(f"Target bucket: gs://{BUCKET_NAME}")

Target project: ai-infra-credit-risk
Target bucket: gs://credit-risk-raw-data


In [ ]:
## Creating gcs bucket
def create_gcs_bucket(bucket_name, project_id, location="EU"):
    """Create a GCS bucket if it does not already exist."""
    client = storage.Client(project=project_id)

    if client.lookup_bucket(bucket_name):
        print(f"Bucket 'gs://{bucket_name}' already exists.")
        return

    bucket = client.bucket(bucket_name)
    bucket.iam_configuration.uniform_bucket_level_access_enabled = True
    bucket = client.create_bucket(bucket, location=location)
    print(f"Successfully created bucket: gs://{bucket.name}")


create_gcs_bucket(BUCKET_NAME, PROJECT_ID, LOCATION)

Successfully created bucket: gs://credit-risk-raw-data


### Importing raw files
I will be directly importing raw csv files from kaggle

In [ ]:
## impoting dataset from kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sergionefedov/credit-risk-dataset-50k-loans-10-sectors")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'credit-risk-dataset-50k-loans-10-sectors' dataset.
Path to dataset files: /kaggle/input/credit-risk-dataset-50k-loans-10-sectors


In [ ]:
## Get all the csv's in the file path
for file_name in os.listdir(path):
 print("FILE =", file_name)

FILE = macro_stress_scenarios.csv
FILE = payment_history.csv
FILE = portfolio_metrics.csv
FILE = payment_history_dictionary.csv
FILE = loan_portfolio.csv
FILE = credit_ratings.csv
FILE = vintage_analysis.csv


### Uploading raw files into gcs bucket

In [ ]:
## uploading all csv's into the gcs bucket
def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):

    print("SOURCE =", source_file_name)
    print("DESTINATION =", destination_blob_name)

    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)

    blob.upload_from_filename(source_file_name)

    print("DONE")

In [ ]:
## uploading all csv's into the gcs bucket
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

for file_name in csv_files:

    print(f"\nPROCESSING {file_name}")

    upload_to_gcs(
        BUCKET_NAME,
        os.path.join(path, file_name),
        f"raw/{file_name}"
    )


PROCESSING macro_stress_scenarios.csv
SOURCE = /kaggle/input/credit-risk-dataset-50k-loans-10-sectors/macro_stress_scenarios.csv
DESTINATION = raw/macro_stress_scenarios.csv
DONE

PROCESSING payment_history.csv
SOURCE = /kaggle/input/credit-risk-dataset-50k-loans-10-sectors/payment_history.csv
DESTINATION = raw/payment_history.csv
DONE

PROCESSING portfolio_metrics.csv
SOURCE = /kaggle/input/credit-risk-dataset-50k-loans-10-sectors/portfolio_metrics.csv
DESTINATION = raw/portfolio_metrics.csv
DONE

PROCESSING payment_history_dictionary.csv
SOURCE = /kaggle/input/credit-risk-dataset-50k-loans-10-sectors/payment_history_dictionary.csv
DESTINATION = raw/payment_history_dictionary.csv
DONE

PROCESSING loan_portfolio.csv
SOURCE = /kaggle/input/credit-risk-dataset-50k-loans-10-sectors/loan_portfolio.csv
DESTINATION = raw/loan_portfolio.csv
DONE

PROCESSING credit_ratings.csv
SOURCE = /kaggle/input/credit-risk-dataset-50k-loans-10-sectors/credit_ratings.csv
DESTINATION = raw/credit_ratings.c

In [ ]:
df_loan = pd.read_csv(f"{path}/loan_portfolio.csv")

df_loan.head()
print(df_loan.columns.tolist())

['loan_id', 'origination_date', 'maturity_date', 'maturity_months', 'sector', 'loan_type', 'collateral', 'initial_rating', 'credit_score', 'ead', 'coupon_rate', 'leverage', 'interest_coverage', 'debt_to_equity', 'pd_annual', 'lgd', 'el', 'unexpected_loss', 'rwa', 'defaulted', 'default_date', 'survival_months', 'recovery_rate', 'loss_given_default']


### Load raw files from GCS to Bigquery

In [ ]:
# -----------------------------
# Option A: Load into BigQuery
# -----------------------------
bq_client = bigquery.Client(project=PROJECT_ID)

# Create dataset
dataset_id = f"{PROJECT_ID}.credit_risk_dataset_raw"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "EU"
dataset = bq_client.create_dataset(dataset, exists_ok=True)

In [ ]:
## Loading all files from GCS to BigQuery
from google.cloud import storage

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

csv_files = []

for blob in bucket.list_blobs(prefix="raw/"):

    if blob.name.endswith(".csv"):

        file_name = blob.name.split("/")[-1]
        csv_files.append(file_name)

print("Files found in GCS:")
for file in csv_files:
    print(file)


Files found in GCS:
credit_ratings.csv
loan_portfolio.csv
macro_stress_scenarios.csv
payment_history.csv
payment_history_dictionary.csv
portfolio_metrics.csv
vintage_analysis.csv


In [ ]:
## Loading all files from GCS to BigQuery
for file_name in csv_files:

    table_name = file_name.replace(".csv", "")
    table_id = f"{dataset_id}.{table_name}"

    uri = f"gs://{BUCKET_NAME}/raw/{file_name}"

    print(f"Loading {file_name}")

    load_job = bq_client.load_table_from_uri(
        uri,
        table_id,
        job_config=bigquery.LoadJobConfig(
            autodetect=True,
            skip_leading_rows=1,
            source_format=bigquery.SourceFormat.CSV,
        ),
    )

    load_job.result()

    print(f"Loaded {table_name}")

Loading credit_ratings.csv
Loaded credit_ratings
Loading loan_portfolio.csv
Loaded loan_portfolio
Loading macro_stress_scenarios.csv
Loaded macro_stress_scenarios
Loading payment_history.csv
Loaded payment_history
Loading payment_history_dictionary.csv
Loaded payment_history_dictionary
Loading portfolio_metrics.csv
Loaded portfolio_metrics
Loading vintage_analysis.csv
Loaded vintage_analysis


### Data SQL transformation and feature engineering
The data from relevant files will be selected and stored as one single table

In [ ]:
## Create a processed table using sql

query = f"""
CREATE OR REPLACE TABLE `{dataset_id}.credit_risk_dataset_processed` AS
SELECT
    loan.*,
    stress.* EXCEPT (sector),
    portf.* EXCEPT (date)
FROM
    `{dataset_id}.loan_portfolio` AS loan
INNER JOIN
    `{dataset_id}.macro_stress_scenarios` AS stress
    ON loan.sector = stress.sector
LEFT JOIN
    `{dataset_id}.portfolio_metrics` AS portf
    ON loan.origination_date = portf.date
"""
# Execute the query
job = bq_client.query(query)
job.result()

print("Processed table created successfully.")

BadRequest: 400 CREATE TABLE has columns with duplicate name credit_spread_bps at [3:1]; reason: invalidQuery, location: query, message: CREATE TABLE has columns with duplicate name credit_spread_bps at [3:1]

Location: EU
Job ID: b241a102-e3c7-42ca-b06f-59e9eee0ddaf


In [ ]:
## Verify the Table
query = """
SELECT COUNT(*)
FROM `YOUR_PROJECT.credit_risk.processed_loans`
"""

df = bq_client.query(query).to_dataframe()

print(df)

### Processed dataset into GCS
The processed dataset will be stored in GCS as a Parquet file

In [ ]:
## Export processed table to Parquet
from google.cloud import bigquery

table_id = f"{dataset_id}.credit_risk_dataset_processed"

destination_uri = (
    f"gs://{BUCKET_NAME}/processed/credit_risk_data_processed.parquet"
)

job_config = bigquery.job.ExtractJobConfig(
    destination_format="PARQUET"
)

extract_job = bq_client.extract_table(
    table_id,
    destination_uri,
    job_config=job_config
)

extract_job.result()

print("Parquet exported successfully.")